<a href="https://colab.research.google.com/github/pnngocdoan/agent-history-effects/blob/main/notebooks/run_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run one model

1. Put your name as DRI on a row of the model table in the research doc.
2. Runtime → Change runtime type → a GPU (L4 or A100 for anything that is not Qwen3).
3. Fill in **cell 3** (family, Hugging Face id, short name), then Runtime → Run all.

The notebook starts the model, runs a 3-conversation pilot and stops if anything looks broken,
then runs the full experiment. Results are pushed to GitHub every 10 minutes and the model shows
up in `reports/viewer.html` under its family on its own. Rerunning only fills in what is missing.

You need, once: push access to this repo and a GitHub token; for gated models (e.g. Gemma), a
Hugging Face token after accepting the model's license on its page.

In [ ]:
# 1. setup (once per runtime)
import os, sys
REPO = '/content/agent-history-effects'
if not os.path.exists(REPO):
    !git clone https://github.com/pnngocdoan/agent-history-effects {REPO}
%cd {REPO}
!git pull
sys.path.insert(0, REPO)
!pip -q install vllm

In [ ]:
# 2. access (once per runtime; tokens are never stored or printed)
from getpass import getpass
import subprocess

name = input('Your name (for commit messages): ')
email = input('Your GitHub email (or <username>@users.noreply.github.com): ')
token = getpass('GitHub token (repo scope): ')
subprocess.run(['git', 'config', 'user.name', name], cwd=REPO)
subprocess.run(['git', 'config', 'user.email', email], cwd=REPO)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/pnngocdoan/agent-history-effects.git'], cwd=REPO)
del token

hf = getpass('Hugging Face token (leave empty if the model is not gated): ')
if hf:
    os.environ['HF_TOKEN'] = hf
del hf

def push(msg):
    r = subprocess.run(f'git add results reports/setup.json && git commit -m "{msg}" && git pull --rebase && git push',
                       shell=True, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(out[-1] if out else 'nothing to push')

In [ ]:
# 3. YOUR MODEL — the only cell to edit
FAMILY = 'Qwen3'            # one of the families in serve.py: Qwen3, Qwen3.5, Gemma 4, gpt-oss
HF_ID  = 'Qwen/Qwen3-4B'    # the model's Hugging Face id
NAME   = 'qwen3-4b'         # short name: the results folder and the tab in the viewer

ROUNDS   = '0 10'           # k = 0 (baseline, no junk rounds) and k = 10
EPISODES = 10               # conversations per k

In [ ]:
# 4. start the model
import time
import serve

TEMPERATURE = serve.FAMILIES[FAMILY]['temperature']
!pkill -f api_server
time.sleep(5)
proc, log = serve.start_server(HF_ID, NAME, log_dir=f'results/logs/{NAME}', family=FAMILY)
serve.wait_for_server(proc, log=log)

In [ ]:
# 5. pilot: 3 conversations at k = 10 before the full run. Stops here if the model never calls a
#    tool, keeps getting cut off, or asks the user instead of acting.
import json
from pathlib import Path

PILOT = f'results/pilot/{NAME}'
!python multiturn/run_pilot.py --model {NAME} --family "{FAMILY}" --rounds 10 --episodes 3 \
    --temperature {TEMPERATURE} --workers 3 --out {PILOT}

rows = [json.loads(l) for l in Path(PILOT, 'episodes.jsonl').read_text().splitlines() if l.strip()]
calls = sum(1 for r in rows for m in r['messages'] if m.get('tool_calls'))
problems = []
if len(rows) < 3:                       problems.append(f'only {len(rows)}/3 conversations finished')
if calls == 0:                          problems.append('no tool calls at all: check the tool-call parser for this family in serve.py')
if sum(r['truncated'] for r in rows):   problems.append('turns were cut off at the token limit')
if sum(r['asked'] for r in rows) == len(rows): problems.append('every conversation ended by asking the user')
if problems:
    raise SystemExit('pilot failed, not starting the full run:\n- ' + '\n- '.join(problems))
print(f'pilot ok: {len(rows)} conversations, {calls} turns with tool calls')

In [ ]:
# 6. background saver: pushes results every 10 minutes while the run goes, so a dead runtime
#    loses at most 10 minutes of conversations
import threading

def _saver():
    while not _stop.is_set():
        _stop.wait(600)
        push(f'autosave: {NAME} in progress')

_stop = threading.Event()
threading.Thread(target=_saver, daemon=True).start()
print('autosave armed (every 10 min)')

In [ ]:
# 7. THE RUN — rerunning only fills in what is missing
OUT = f'results/multiturn/{NAME}'
!python multiturn/run_pilot.py --model {NAME} --family "{FAMILY}" --rounds {ROUNDS} \
    --episodes {EPISODES} --temperature {TEMPERATURE} --workers 8 --out {OUT}
_stop.set()
proc.terminate()
push(f'results: {NAME}')
print(f'done: {NAME} is on GitHub and in reports/viewer.html under {FAMILY}')